In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.metrics import classification_report, confusion_matrix
import lightgbm as lgb
import tensorflow as tf

In [30]:
# Paths
BASE_PATH = "/content/drive/MyDrive/Zero-Day-IoT-Project"

DATA_PATH   = BASE_PATH + "/01_Data"
MODEL_PATH  = BASE_PATH + "/03_Models"
REPORT_PATH = BASE_PATH + "/04_Reports"

os.makedirs(REPORT_PATH, exist_ok=True)

In [31]:
# Load Models
lgb_model = joblib.load(MODEL_PATH + "/lightgbm_known.pkl")

autoencoder = tf.keras.models.load_model(
    MODEL_PATH + "/autoencoder.keras"
)

threshold = joblib.load(MODEL_PATH + "/ae_threshold.pkl")

scaler = joblib.load(MODEL_PATH + "/scaler.pkl")

print("Models Loaded")

Models Loaded


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 14 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [32]:
# Load full dataset
full_df = pd.read_parquet(DATA_PATH + "/master_clean.parquet")

# Normal traffic
normal_df = full_df[full_df['Attack_label'] == 0]

# Zero-day attack dataset
zero_df = pd.read_parquet(DATA_PATH + "/zero_day_test.parquet")

# Combine both
df = pd.concat([normal_df, zero_df], axis=0)

print("Final Test Dataset Distribution:\n")
print(df['Attack_label'].value_counts())

Final Test Dataset Distribution:

Attack_label
1    31096
0    24152
Name: count, dtype: int64


In [33]:
# Prepare Features
feature_cols = [c for c in df.columns if c not in ['Attack_label', 'Attack_type']]

X = df[feature_cols]
y_true = df['Attack_label']

In [34]:
# LightGBM prediction
lgb_probs = lgb_model.predict_proba(X)

lgb_pred_class = np.argmax(lgb_probs, axis=1)

lgb_confidence = np.max(lgb_probs, axis=1)

In [35]:
# Autoencoder predictions
X_scaled = scaler.transform(X)

X_recon = autoencoder.predict(X_scaled)

recon_error = np.mean(np.square(X_scaled - X_recon), axis=1)

ae_pred = (recon_error > threshold).astype(int)

1727/1727 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


In [36]:
# Fusion
CONF_THRESHOLD = 0.90

final_pred = []

for i in range(len(X)):

    # If LightGBM is VERY confident it's NORMAL (class 0)
    if lgb_pred_class[i] == 0 and lgb_confidence[i] >= CONF_THRESHOLD:
        final_pred.append(0)   # Normal

    else:
        # Otherwise rely on Autoencoder
        final_pred.append(ae_pred[i])

final_pred = np.array(final_pred)

In [37]:
# Evaluation
print("FUSION MODEL RESULTS\n")

print(classification_report(y_true, final_pred))

FUSION MODEL RESULTS

              precision    recall  f1-score   support

           0       0.98      0.95      0.96     24152
           1       0.96      0.98      0.97     31096

    accuracy                           0.97     55248
   macro avg       0.97      0.97      0.97     55248
weighted avg       0.97      0.97      0.97     55248



In [38]:
# Confusion Matrix
cm = confusion_matrix(y_true, final_pred)

print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[22913  1239]
 [  520 30576]]


In [39]:
# Save Results
report = classification_report(y_true, final_pred, output_dict=True)

pd.DataFrame(report).transpose().to_csv(
    REPORT_PATH + "/fusion_model_metrics.csv"
)

print("Fusion Results Saved")

Fusion Results Saved


In [40]:
# Compare Models
print("Model Comparison Summary")

print("LightGBM: 99.99% (Known attacks)")
print("Autoencoder: ~99% (Anomaly detection)")
print("Fusion Model: Combined intelligence")

Model Comparison Summary
LightGBM: 99.99% (Known attacks)
Autoencoder: ~99% (Anomaly detection)
Fusion Model: Combined intelligence
